In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv") # We initialize the complete path with Q1_data.csv
df = pd.read_csv(csv_path) # Now We can read the csv file we prepared

In [ ]:
# Task 2: Write your code here:
df.head() #printing the first 5 samples with the features

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
# df.head() # to check that the column is deleted

In [ ]:
# Task 2: Write your code here:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print("tot:", (df.isnull().sum().sum() / len(df)) * 100) # 18% is a lot so we will not drop the columns
# For objects (text) I used the most frequent with None
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
# Since the the plot is balanced and makes a bell shape then I will use mean, if the plot was skewed i will use median
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

# Now there is no missing values.

In [ ]:
# Task 3: Write your code here:
df.drop_duplicates(inplace=True) # inplace will modify the origianl df
print(f"Duplicates: {df.duplicated().sum()}")
#Now all duplicates are removed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
#The following columns has no order so we use OneHotEncoder. + The columns have few unique samples, if many wouldn't use it
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df[['Weather']])
df["Weather"] = encoded
encoded = encoder.fit_transform(df[['Vehicle_Type']])
df["Vehicle_Type"] = encoded

# When order matters daytime and time of the day is ordered
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))
df.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
# The target should not be scaled we need to seperate them.

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 6: Write your code here:
# The data before adding the mean to nulls were balanced, but after adding the mean to nulls that data showed a huge spike in mean
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

avg = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
 X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
 y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
 rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
 rf.fit(X_train, y_train)
 y_pred = rf.predict(X_val)
 mae = mean_absolute_error(y_val, y_pred)
 avg.append(mae)

print(np.average(avg))





In [ ]:
# Task 1: Write your code here:
print("\nTop important features (Random Forest):")
importances = rf.feature_importances_
for i in np.argsort(importances)[::-1]:
    print(f"  Feature {i}: {importances[i]:.3f}")

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(importances, bins=10, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: